# Inference and Sampling


Use this notebook after the inference and sampling note. The goal is to separate the model's logits from the decoding rule that chooses the next token.

You will:
- run prefill and decode with a cache
- compare greedy, temperature, top-k, and top-p behavior
- connect the small teaching runtime to the fuller picoLLM serving controls


In [1]:
import sys
from pathlib import Path
import torch

ROOT = next((path for path in [Path.cwd(), *Path.cwd().parents] if (path / 'course_tools').exists() or (path / 'picollm').exists()), Path.cwd())
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from pathlib import Path
import torch
from course_tools import DEFAULT_CHAT_MESSAGES, ensure_demo_checkpoint, format_messages, generate_text, prefill_prompt, decode_next_token

LECTURE_NOTE_TITLE = 'Inference and Sampling'
print(f'Lecture note: {LECTURE_NOTE_TITLE}')
torch.manual_seed(0)
bundle = ensure_demo_checkpoint(steps=40)
model = bundle['model']
tokenizer = bundle['tokenizer']
prompt = format_messages(DEFAULT_CHAT_MESSAGES, add_assistant_prompt=True)


Lecture note: Inference and Sampling


## Prefill then decode


In [2]:
prompt_ids, past_kvs = prefill_prompt(model, tokenizer, prompt)
print('prefill tokens:', len(prompt_ids))
print('cache shapes:', [kv[0].shape for kv in past_kvs])
next_id, past_kvs, _ = decode_next_token(model, prompt_ids[-1], past_kvs, temperature=0.8, top_k=8)
print('first decoded token:', next_id, repr(tokenizer.decode([next_id])))


prefill tokens: 64
cache shapes: [torch.Size([1, 4, 64, 12]), torch.Size([1, 4, 64, 12])]
first decoded token: 35 'n'


## Greedy decoding


In [3]:
print(repr(generate_text(model, tokenizer, prompt, max_new_tokens=60, temperature=0.0, top_k=None)))


'nintintinininininininininininininininininininininininininini'


## Temperature and top-k on the same logits


In [4]:
for temperature, top_k in [(0.4, 4), (0.8, 8), (1.2, 12)]:
    print({'temperature': temperature, 'top_k': top_k})
    print(repr(generate_text(model, tokenizer, prompt, max_new_tokens=60, temperature=temperature, top_k=top_k)))
    print('-' * 80)


{'temperature': 0.4, 'top_k': 4}
'ntintintiniintinintins tintintininininintintinins tintinnini'
--------------------------------------------------------------------------------
{'temperature': 0.8, 'top_k': 8}
' tetnin ctintintitttrnssidiotininiiiitniinintssininntiniitii'
--------------------------------------------------------------------------------
{'temperature': 1.2, 'top_k': 12}
'\nneensaac anntnininntismderinncitin adn cnrtitaldinonctioths'
--------------------------------------------------------------------------------


## Top-p as a common extension


In [5]:
def top_p_candidates(logits: torch.Tensor, p: float) -> tuple[torch.Tensor, torch.Tensor, torch.Tensor]:
    probs = torch.softmax(logits, dim=-1)
    sorted_probs, sorted_ids = torch.sort(probs, descending=True)
    cumulative = torch.cumsum(sorted_probs, dim=-1)
    crossing_index = int(torch.searchsorted(cumulative, torch.tensor(p), right=False).item())
    crossing_index = min(crossing_index, len(sorted_probs) - 1)
    keep = torch.zeros_like(cumulative, dtype=torch.bool)
    keep[: crossing_index + 1] = True
    return sorted_ids[keep], sorted_probs[keep], cumulative[keep]


toy_tokens = ["the", "a", "red", "blue", "cat", "spaceship", "<eos>"]
toy_logits = torch.tensor([4.8, 4.0, 2.6, 2.1, 1.2, 0.4, -0.5])
base_probs = torch.softmax(toy_logits, dim=-1)

print("full distribution:")
for token, prob in zip(toy_tokens, base_probs.tolist()):
    print(f"{token:>10s}  {prob:.4f}")

print()
print("top-p keeps the smallest sorted set whose cumulative probability reaches p:")
for p in [0.50, 0.80, 0.95]:
    ids, probs, cumulative = top_p_candidates(toy_logits, p)
    kept = [(toy_tokens[i], round(float(prob), 4), round(float(cum), 4)) for i, prob, cum in zip(ids.tolist(), probs, cumulative)]
    print({"top_p": p, "kept": kept})

print()
print("small teaching runtime: temperature + top_k")
print("serious picoLLM runtime: temperature + top_k + top_p + min_p + max_tokens + seed")


full distribution:
       the  0.5981
         a  0.2687
       red  0.0663
      blue  0.0402
       cat  0.0163
 spaceship  0.0073
     <eos>  0.0030

top-p keeps the smallest sorted set whose cumulative probability reaches p:
{'top_p': 0.5, 'kept': [('the', 0.5981, 0.5981)]}
{'top_p': 0.8, 'kept': [('the', 0.5981, 0.5981), ('a', 0.2687, 0.8669)]}
{'top_p': 0.95, 'kept': [('the', 0.5981, 0.5981), ('a', 0.2687, 0.8669), ('red', 0.0663, 0.9331), ('blue', 0.0402, 0.9733)]}

small teaching runtime: temperature + top_k
serious picoLLM runtime: temperature + top_k + top_p + min_p + max_tokens + seed


## External reference repos


**RASBT**
- https://github.com/rasbt/LLMs-from-scratch/blob/main/ch05/01_main-chapter-code/gpt_generate.py
**NANOCHAT**
- https://github.com/karpathy/nanochat/blob/master/nanochat/engine.py
